# 07, Knowledge chunking, embeddings, and GraphRAG

**Inputs:** `config/seed/knowledge/*.md` (the 6 seeded SOPs), `config/seed/graph_seed.yaml`

**Outputs:** chunking-strategy comparison, embedding-model comparison (recall@5, MRR on 40
hand-written question -> chunk pairs), and a dense-only vs dense+graph-expansion comparison on
fault prediction, the experiment that justifies ADR-007 (graph expansion on top of dense
retrieval).

**Substitution note:** the plan calls for MiniLM-L6 vs BGE-small; this environment substitutes
`all-MiniLM-L12-v2` for BGE-small (both ~small, CPU-fast, no gated/large download) so the
comparison methodology runs end to end here. Swap in `BAAI/bge-small-en-v1.5` for a plan-faithful
run where a larger download is acceptable.

In [1]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

cwd: D:\DSEP22\v1 | APP_PROFILE: stub


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

from libs.platform.graph.inmemory import InMemoryGraphStore
from services.knowledge_ingest.chunk import heading_aware_chunks
from services.retrieval_svc.handler import _extract_seed_terms

KNOWLEDGE_DIR = Path("config/seed/knowledge")
DOC_PATHS = sorted(KNOWLEDGE_DIR.glob("*.md"))
[p.name for p in DOC_PATHS]

D:\DSEP22\v1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['sop_account_retention.md',
 'sop_billing.md',
 'sop_field_service_visit.md',
 'sop_intermittent.md',
 'sop_line_sync.md',
 'sop_router_power.md']

## Chunking-strategy comparison: fixed 500/800/1200 chars vs heading-aware

In [3]:
def fixed_chunks(text: str, size: int) -> list[str]:
    text = text.strip()
    return [text[i : i + size].strip() for i in range(0, len(text), size) if text[i : i + size].strip()]


rows = []
for path in DOC_PATHS:
    text = path.read_text(encoding="utf-8")
    for strategy, chunks in [
        ("fixed_500", fixed_chunks(text, 500)),
        ("fixed_800", fixed_chunks(text, 800)),
        ("fixed_1200", fixed_chunks(text, 1200)),
        ("heading_aware", heading_aware_chunks(text)),
    ]:
        lengths = [len(c) for c in chunks]
        rows.append({
            "doc": path.stem, "strategy": strategy, "num_chunks": len(chunks),
            "avg_len": sum(lengths) / len(lengths) if lengths else 0,
        })

import pandas as pd
chunking_df = pd.DataFrame(rows)
chunking_df.groupby("strategy")[["num_chunks", "avg_len"]].mean()

,num_chunks,avg_len
strategy,,
fixed_1200,1.666667,830.750000
fixed_500,3.166667,423.652778
fixed_800,2.000000,658.916667
heading_aware,5.666667,231.674206


Heading-aware chunking produces more, shorter chunks than the fixed strategies because it never
merges two `##` sections together, even when both are under the size threshold, e.g. a short
"Symptoms" section becomes its own chunk instead of bleeding into "Diagnosis". This is exactly
the property that makes citations precise (retrieval points at one procedure, not two).

## Build the real chunk set (heading-aware) and embed with two models

In [4]:
doc_chunks: dict[str, list[str]] = {}
for path in DOC_PATHS:
    doc_chunks[path.stem] = heading_aware_chunks(path.read_text(encoding="utf-8"))

all_chunks = [(doc, i, text) for doc, chunks in doc_chunks.items() for i, text in enumerate(chunks)]
print(f"{len(all_chunks)} chunks across {len(doc_chunks)} documents")

model_a = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model_b = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")

texts = [c[2] for c in all_chunks]
emb_a = model_a.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
emb_b = model_b.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

34 chunks across 6 documents


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17680.83it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8653.79it/s]

## 40 hand-written question -> chunk pairs

One or more questions per chunk, phrased the way a support agent would actually ask (not just
the section heading reworded), covering all 6 SOPs.

In [5]:
# (doc stem, chunk index, question)
QA_PAIRS = [
    ("sop_router_power", 1, "What are the symptoms of a router power issue?"),
    ("sop_router_power", 1, "My router's power LED is blinking red, what does that mean?"),
    ("sop_router_power", 2, "How do I diagnose whether it's a power supply fault?"),
    ("sop_router_power", 3, "How do I power-cycle the router?"),
    ("sop_router_power", 3, "What are the steps to restart the router?"),
    ("sop_router_power", 4, "How do I factory reset the router?"),
    ("sop_router_power", 5, "When should I send a replacement power supply?"),
    ("sop_router_power", 6, "When do I escalate a router power issue to field service?"),
    ("sop_line_sync", 1, "What does it mean if the internet LED is amber?"),
    ("sop_line_sync", 1, "The internet light is off completely, what's wrong?"),
    ("sop_line_sync", 2, "How do I tell if it's a line synchronisation fault?"),
    ("sop_line_sync", 3, "How do I check the WAN cabling?"),
    ("sop_line_sync", 4, "How do I run a remote line diagnostic?"),
    ("sop_line_sync", 4, "What does the run_line_diagnostic action check?"),
    ("sop_line_sync", 5, "When does a line sync issue escalate to network operations?"),
    ("sop_intermittent", 1, "The connection drops for a few seconds at a time, why?"),
    ("sop_intermittent", 1, "Wifi disconnects randomly during the day, what could cause that?"),
    ("sop_intermittent", 2, "How do I figure out if intermittent drops are congestion or cabling?"),
    ("sop_intermittent", 3, "What diagnostic should I run for intermittent connection loss?"),
    ("sop_intermittent", 4, "When should I book a technician for intermittent drops?"),
    ("sop_intermittent", 5, "When does an intermittent connection issue escalate further?"),
    ("sop_billing", 1, "The customer says they were charged twice, what's the process?"),
    ("sop_billing", 1, "A customer was billed after they already cancelled, what do I check?"),
    ("sop_billing", 2, "How do I check if a billing dispute is genuine?"),
    ("sop_billing", 3, "How do I issue a billing credit?"),
    ("sop_billing", 3, "What reason codes are used for billing credits?"),
    ("sop_billing", 4, "The customer says their payment isn't showing, what do I do?"),
    ("sop_billing", 5, "What must I avoid promising in a billing dispute reply?"),
    ("sop_field_service_visit", 1, "When do I schedule a field service visit instead of a remote fix?"),
    ("sop_field_service_visit", 2, "How do I book a technician visit?"),
    ("sop_field_service_visit", 2, "What address details do I need to schedule an engineer visit?"),
    ("sop_field_service_visit", 3, "What happens if a second technician visit is needed within 14 days?"),
    ("sop_account_retention", 1, "A customer wants to cancel their subscription, what should I check first?"),
    ("sop_account_retention", 1, "The customer is threatening to switch providers, what do I do?"),
    ("sop_account_retention", 2, "How do I handle a cancellation request?"),
    ("sop_account_retention", 2, "Can I offer a discount to stop someone cancelling?"),
    ("sop_account_retention", 3, "What must I avoid saying about a cancellation that hasn't been processed?"),
    ("sop_router_power", 0, "What is this document about?"),
    ("sop_line_sync", 0, "What SOP covers line synchronisation problems?"),
    ("sop_billing", 0, "What SOP covers billing disputes?"),
]
len(QA_PAIRS)

40

## recall@5 and MRR for each embedding model

In [6]:
chunk_index_lookup = {(doc, i): pos for pos, (doc, i, _) in enumerate(all_chunks)}


def evaluate(embeddings: np.ndarray) -> dict:
    hits_at_5 = 0
    reciprocal_ranks = []
    for doc, chunk_idx, question in QA_PAIRS:
        target_pos = chunk_index_lookup.get((doc, chunk_idx))
        if target_pos is None:
            continue
        q_emb = model_a.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0] \
            if embeddings is emb_a else model_b.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
        sims = embeddings @ q_emb
        ranked = np.argsort(-sims)
        rank = int(np.where(ranked == target_pos)[0][0]) + 1
        hits_at_5 += int(rank <= 5)
        reciprocal_ranks.append(1.0 / rank)

    n = len(QA_PAIRS)
    return {"recall_at_5": hits_at_5 / n, "mrr": sum(reciprocal_ranks) / n}


results_a = evaluate(emb_a)
results_b = evaluate(emb_b)
print("all-MiniLM-L6-v2: ", results_a)
print("all-MiniLM-L12-v2:", results_b)

all-MiniLM-L6-v2:  {'recall_at_5': 0.9, 'mrr': 0.6682863408521303}
all-MiniLM-L12-v2: {'recall_at_5': 0.9, 'mrr': 0.657843567251462}


## Dense-only vs dense+graph expansion on fault prediction

For a small set of realistic customer-phrased symptom queries, compare: does the *document* of
the top-1 dense hit correspond to a plausible fault for that symptom, vs does the fault actually
appear as a node when the query's keywords are expanded through the seed graph
(`InMemoryGraphStore.expand`, the mechanism `retrieval_svc` uses in production)?

In [7]:
DOC_TO_PLAUSIBLE_FAULTS = {
    "sop_router_power": {"fault_power_supply", "fault_firmware_crashloop", "fault_hardware"},
    "sop_line_sync": {"fault_line_sync", "fault_cabling"},
    "sop_intermittent": {"fault_intermittent_connection"},
    "sop_billing": {"fault_billing_dispute"},
    "sop_field_service_visit": {"fault_hardware", "fault_cabling", "fault_no_dial_tone"},
}

FAULT_QUERIES = [
    ("my router power light is red and wont turn on", "fault_power_supply"),
    ("the router keeps rebooting itself over and over", "fault_firmware_crashloop"),
    ("internet led stuck amber never goes green", "fault_line_sync"),
    ("connection keeps dropping randomly throughout the day", "fault_intermittent_connection"),
    ("i was charged twice on my invoice this month", "fault_billing_dispute"),
    ("wifi drops out every few minutes all afternoon", "fault_intermittent_connection"),
    ("the power light on my modem wont come on at all", "fault_power_supply"),
]

graph_store = InMemoryGraphStore(Path("config/seed/graph_seed.yaml"))
doc_texts = [" ".join(doc_chunks[doc]) for doc in doc_chunks]
doc_names = list(doc_chunks.keys())
doc_embeddings = model_a.encode(doc_texts, convert_to_numpy=True, normalize_embeddings=True)

dense_correct, graph_correct = 0, 0
for query, expected_fault in FAULT_QUERIES:
    q_emb = model_a.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    top_doc = doc_names[int(np.argmax(doc_embeddings @ q_emb))]
    dense_ok = expected_fault in DOC_TO_PLAUSIBLE_FAULTS.get(top_doc, set())
    dense_correct += int(dense_ok)

    seed_terms = _extract_seed_terms(query)
    subgraph = graph_store.expand(seed_terms, hops=2)
    graph_faults = {n["id"] for n in subgraph.nodes if n.get("label") == "Fault"}
    graph_ok = expected_fault in graph_faults
    graph_correct += int(graph_ok)

    print(f"{query!r:60s} expected={expected_fault:28s} dense_top_doc={top_doc:24s} dense_ok={dense_ok!s:5s} graph_ok={graph_ok}")

n = len(FAULT_QUERIES)
print(f"\ndense-only accuracy: {dense_correct/n:.2f}   dense+graph accuracy: {graph_correct/n:.2f}")

'my router power light is red and wont turn on'              expected=fault_power_supply           dense_top_doc=sop_router_power         dense_ok=True  graph_ok=True
'the router keeps rebooting itself over and over'            expected=fault_firmware_crashloop     dense_top_doc=sop_router_power         dense_ok=True  graph_ok=True
'internet led stuck amber never goes green'                  expected=fault_line_sync              dense_top_doc=sop_line_sync            dense_ok=True  graph_ok=True
'connection keeps dropping randomly throughout the day'      expected=fault_intermittent_connection dense_top_doc=sop_intermittent         dense_ok=True  graph_ok=True
'i was charged twice on my invoice this month'               expected=fault_billing_dispute        dense_top_doc=sop_billing              dense_ok=True  graph_ok=False
'wifi drops out every few minutes all afternoon'             expected=fault_intermittent_connection dense_top_doc=sop_intermittent         dense_ok=True  graph_ok=

**Honest reading of these numbers** (worth stating plainly rather than picking the result that
fits a nicer story): dense-only scores 100% here, not graph. That's because the "dense_ok" metric
is document-level and lenient (does the top-retrieved *document* plausibly contain the expected
fault, out of a per-document candidate set), and with only 6 topically well-separated SOPs, doc
retrieval is an easy task at that granularity.

What the graph actually contributes, and dense-only does not, is **fault-level precision**: for
every symptom-shaped query it resolves, it returns a small, specific candidate set
(`fault_power_supply` + `fault_firmware_crashloop`, not "somewhere in sop_router_power's ~1500
characters"). Its one miss is real and structural, not incidental: `fault_billing_dispute` has no
`LedState`/`Symptom` node feeding it in the seed graph, so keyword-expansion from a billing query
can never reach it, dense retrieval is the only path to billing content. That is the actual
argument for ADR-007's *dense-first, then graph-expand* order: graph narrows dense's output to a
precise fault when the vocabulary lines up with the seed graph, and dense is the fallback (not an
afterthought) for everything the graph has no path to. A same-granularity metric (graph's fault
set vs dense's single nearest chunk mapped to a fault) would be a fairer head-to-head than this
document-level proxy, worth doing once the KB is large enough that document-level retrieval
stops being trivial.

In [8]:
import json

with open("evaluation/reports/metrics.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "notebook": "07_knowledge_and_graphrag",
        "chunking_comparison": chunking_df.groupby("strategy")[["num_chunks", "avg_len"]].mean().to_dict(),
        "embedding_all_minilm_l6": results_a,
        "embedding_all_minilm_l12": results_b,
        "dense_only_fault_accuracy": dense_correct / n,
        "dense_plus_graph_fault_accuracy": graph_correct / n,
    }) + "\n")
print("appended metrics row")

appended metrics row
